# RESCUE-0001 — R3 capped/low-agreement → R2 SC8 conditional rescue

This notebook runs a **dev-only policy gate first**, then applies exactly one frozen policy to the public leaderboard. R3 remains the SC16 base. R2 Pro4-hint is called only when R3 has a 2,048-token cap and is low-agreement (`margin ≤ 1` or `top_count < 8`). No answers are read in leaderboard mode.

In [ ]:
# Cell 1 — Fresh A100 only. Run once, then restart the runtime before Cell 2.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart runtime once, then run Cells 2–6.")


In [ ]:
# Cell 2 — Mount Drive, update the reproducibility repo, and cache the pinned base model.
from google.colab import drive, userdata
from pathlib import Path
import subprocess
drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = "https://github.com/jhparktime/qwen-math-final-2026.git"
if token:
    url = "https://x-access-token:" + token + "@github.com/jhparktime/qwen-math-final-2026.git"
repo = Path("/content/qwen-math-final")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Freeze paths. This uses the R3 SC16 raw file already produced by LB-0016 for the leaderboard.
import re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r"[\s_-]+", "", unicodedata.normalize("NFC", str(value)).casefold())

roots = [p for p in Path("/content/drive/MyDrive").iterdir() if p.is_dir() and compact(p.name) == compact("2026소중한챌린지")]
assert len(roots) == 1, roots
project = roots[0]
runs = project / "runs"
splits = runs / "AUDIT-0002-clean-split-passN-20260821-215706" / "splits"
DEV_PATH = splits / "dev_v1.csv"
LEADERBOARD_PATH = project / "data" / "deep_chal_math_leaderboard_filtered.csv"
R3_ADAPTER = runs / "RFT-0008D-r3mix-r2continue-r16-a100" / "adapter_final"
R2_ADAPTER = runs / "RFT-0004B-r2-pro4-hint-lowdrift-lora" / "adapter_final"
# Generated in LB-0016: it is only reused as R3 SC16 candidates; no answer labels are present.
R3_LEADERBOARD_RAW = runs / "LB-0016-r3-sc16-pal3-adaptive" / "candidates" / "final_test_r2_sc16_2048.jsonl"
RUN_ROOT = runs / "RESCUE-0001-r3cap-r2sc8-devgate"
for path in [DEV_PATH, LEADERBOARD_PATH, R3_ADAPTER / "adapter_config.json", R3_ADAPTER / "adapter_model.safetensors", R2_ADAPTER / "adapter_config.json", R2_ADAPTER / "adapter_model.safetensors", R3_LEADERBOARD_RAW]:
    assert path.exists(), path
print("[DEV]", DEV_PATH)
print("[R3 LB raw]", R3_LEADERBOARD_RAW)
print("[RUN]", RUN_ROOT)


In [ ]:
# Cell 4 — Dev gate. Generates R3 SC16 on fixed dev, then R2 SC8 only on R3 capped + low-agreement rows.
# A deterministic policy/confirmation split selects a policy; do not manually select a different policy afterward.
DEV_OUT = RUN_ROOT / "dev"
!PYTHONPATH=. python3 inference/r3_r2_cap_rescue.py --input {DEV_PATH} --r3-adapter {R3_ADAPTER} --r2-adapter {R2_ADAPTER} --output-dir {DEV_OUT} --split dev


In [ ]:
# Cell 5 — Apply the frozen dev policy to the leaderboard. R3 SC16 is reused; only target rows get R2 SC8.
LB_OUT = RUN_ROOT / "leaderboard"
POLICY_PATH = DEV_OUT / "reports" / "frozen_policy.json"
!PYTHONPATH=. python3 inference/r3_r2_cap_rescue.py --input {LEADERBOARD_PATH} --r3-adapter {R3_ADAPTER} --r2-adapter {R2_ADAPTER} --output-dir {LB_OUT} --split leaderboard --frozen-policy {POLICY_PATH} --reuse-r3-raw {R3_LEADERBOARD_RAW}
SUBMISSION_PATH = LB_OUT / "submissions" / "submission_r3cap_r2rescue.csv"
EASY_COPY = Path("/content/drive/MyDrive/submission_r3cap_r2rescue.csv")
EASY_COPY.write_bytes(SUBMISSION_PATH.read_bytes())
print("[SUBMIT]", EASY_COPY)


In [ ]:
# Cell 6 — Review the frozen gate and final change count before submitting.
import json
print((DEV_OUT / "reports" / "frozen_policy.json").read_text())
print((LB_OUT / "reports" / "leaderboard_report.json").read_text())


In [ ]:
# Final cell — opt-in GPU release.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True when done.")
